# Time series fundamentals

Datetime indexing, frequency, missing timestamps, plots and summary statistics.

Dataset is synthetic. Run cells top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
p = Path("data/daily_store_sales.csv")
if not p.exists(): p = Path("daily_store_sales.csv")
df = pd.read_csv(p, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
print(df.head())
print("Rows:", len(df), "Duplicate dates:", df.date.duplicated().sum())

In [ ]:
print(df.sales.describe())
ax=df.plot(x="date", y="sales", figsize=(11,3), title="Daily sales (synthetic)")
plt.tight_layout(); plt.show()

In [ ]:
df["rolling_7"] = df.sales.rolling(7).mean()
ax=df.plot(x="date", y=["sales","rolling_7"], figsize=(11,3), title="Trend smoothing")
plt.tight_layout();plt.show()

In [ ]:
print("Mean sales by weekend flag:\n",df.groupby("weekend").sales.mean())
print("Monthly average:\n",df.groupby(df.date.dt.month).sales.mean())

In [ ]:
cut=int(len(df)*.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
actual=test.sales.to_numpy()
last=np.repeat(train.sales.iloc[-1],len(test))
seasonal=np.resize(train.sales.iloc[-7:].to_numpy(),len(test))
def score(name,pred):
    return {"model":name,"MAE":mean_absolute_error(actual,pred),"RMSE":np.sqrt(mean_squared_error(actual,pred))}
results=[score("last value",last),score("seasonal naive (7d)",seasonal)]
print(pd.DataFrame(results).to_string(index=False))

In [ ]:
expected=pd.date_range(df.date.min(),df.date.max(),freq="D")
print("Missing dates:",len(expected.difference(df.date)))
print("Sales by day of week:",df.groupby(df.date.dt.day_name()).sales.mean().round(1))

## Interpretation

Compare errors to a simple baseline. A smaller MAE is better, but this synthetic example cannot establish deployment performance. Inspect residuals, calendar changes, and whether covariates are available in advance.